# Week 4: Backpropagation, SGD, and Regularization

**Lecture 7:** Backpropagation implementation  
**Lecture 8:** Stochastic gradient descent and regularization

# Lecture 7: Implementing Backpropagation

This implementation follows the equations from Week 3. The forward pass stores every activation. The backward pass computes the output delta, propagates deltas backward, and constructs explicit gradients for each $W^{(\ell)}$ and $b^{(\ell)}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
class MultilayerPerceptron:
    def __init__(
        self,
        layer_sizes,
        alpha=0.5,
        batch_size=None,
        l2=0.0,
        seed=1,
    ):
        self.alpha = alpha
        self.batch_size = batch_size
        self.l2 = l2
        self.rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []

        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:]):
            scale = np.sqrt(2 / (input_size + output_size))
            self.weights.append(
                self.rng.normal(0, scale, size=(input_size, output_size))
            )
            self.biases.append(np.zeros(output_size))

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def sigmoid_gradient(a):
        return a * (1 - a)

    def forward(self, X):
        activations = [np.atleast_2d(X)]

        for W, b in zip(self.weights, self.biases):
            z = activations[-1] @ W + b
            activations.append(self.sigmoid(z))

        return activations

    def loss(self, X, y):
        y = np.atleast_2d(y).reshape(-1, 1)
        y_hat = self.forward(X)[-1]
        data_loss = 0.5 * np.mean((y_hat - y) ** 2)
        penalty = 0.5 * self.l2 * sum(
            np.sum(W**2) for W in self.weights
        )
        return data_loss + penalty

    def gradients(self, X, y):
        X = np.atleast_2d(X)
        y = np.atleast_2d(y).reshape(-1, 1)
        n = X.shape[0]
        activations = self.forward(X)

        weight_gradients = [None] * len(self.weights)
        bias_gradients = [None] * len(self.biases)

        delta = (activations[-1] - y) * self.sigmoid_gradient(activations[-1])

        for layer in range(len(self.weights) - 1, -1, -1):
            weight_gradients[layer] = (
                activations[layer].T @ delta / n + self.l2 * self.weights[layer]
            )
            bias_gradients[layer] = delta.mean(axis=0)

            if layer > 0:
                delta = (
                    delta @ self.weights[layer].T
                    * self.sigmoid_gradient(activations[layer])
                )

        return weight_gradients, bias_gradients

    def fit(self, X, y, epochs=1_000, update=100):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        batch_size = self.batch_size or len(X)
        history = []

        for epoch in range(epochs):
            indices = self.rng.permutation(len(X))

            for start in range(0, len(X), batch_size):
                batch = indices[start : start + batch_size]
                weight_gradients, bias_gradients = self.gradients(
                    X[batch], y[batch]
                )

                for layer in range(len(self.weights)):
                    self.weights[layer] -= self.alpha * weight_gradients[layer]
                    self.biases[layer] -= self.alpha * bias_gradients[layer]

            if (epoch + 1) % update == 0 or epoch == 0:
                history.append((epoch + 1, self.loss(X, y)))

        return history

    def predict_proba(self, X):
        return self.forward(X)[-1]

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int).ravel()

### XOR Example

With `batch_size=None`, each update uses the entire dataset: ordinary gradient descent. XOR verifies that a hidden layer can learn a nonlinear decision rule.

In [ ]:
X_xor = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y_xor = np.array([0, 1, 1, 0])

xor_model = MultilayerPerceptron(
    [2, 4, 1], alpha=1.0, batch_size=None, seed=4
)
history = xor_model.fit(X_xor, y_xor, epochs=10_000, update=1_000)

print("training history:", history)
print("probabilities:", np.round(xor_model.predict_proba(X_xor).ravel(), 3))
print("predictions:", xor_model.predict(X_xor))

# Lecture 8: Stochastic Gradient Descent and Regularization

Full-batch gradient descent computes one update from all $n$ training examples. **Stochastic gradient descent (SGD)** estimates the gradient from a mini-batch $B$:

$$
$$\nabla L_B(w)=\frac{1}{|B|}\sum_{i\in B}\nabla L_i(w).$$

Smaller batches produce cheaper, noisier, and more frequent updates.

Regularization discourages a model from fitting noise. With L2 regularization (weight decay),

$$L_{\mathrm{regularized}}(w)=L_{\mathrm{data}}(w)+\frac{\lambda}{2}\sum_{\ell}\|W^{(\ell)}\|_F^2,$$

so each weight gradient gains the explicit term $\lambda W^{(\ell)}$. Biases are not penalized here. Other common strategies include early stopping, dropout, and data augmentation.

### Mini-batch SGD Example

The same implementation now uses mini-batches and L2 weight decay on a nonlinear two-class dataset.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.25, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MultilayerPerceptron(
    [2, 16, 8, 1],
    alpha=0.5,
    batch_size=32,
    l2=1e-4,
    seed=1,
)
history = model.fit(X_train, y_train, epochs=2_000, update=100)

print("final recorded loss:", round(history[-1][1], 6))
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200),
)
grid = np.column_stack([xx.ravel(), yy.ravel()])
probabilities = model.predict_proba(grid).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, probabilities, levels=20, cmap="RdBu", alpha=0.7)
plt.scatter(
    X_train[:, 0], X_train[:, 1], c=y_train, cmap="RdBu", edgecolor="white"
)
plt.title("Mini-batch SGD with L2 regularization")
plt.xlabel("standardized feature 1")
plt.ylabel("standardized feature 2")
plt.show()